# Mixture of Experts：从路由公式到系统瓶颈

这个 notebook 用 NumPy 实现一个可观察的稀疏 MoE 前向过程，重点不是训练出大模型，而是把 router、top-k、dispatch、expert、combine、capacity 和负载指标逐层拆开。

## 学习目标

1. 区分总参数、激活参数、理论 FLOPs 与真实延迟。
2. 手写 `router -> top-k -> dispatch -> expert -> combine`。
3. 理解 capacity factor、token drop、负载均衡损失与 expert collapse。
4. 解释 expert parallel 为什么需要 all-to-all。
5. 能设计 MoE 的训练与线上监控指标。

## 1. Dense FFN 与稀疏 MoE

Dense FFN 对所有 token 使用同一套参数：

$$y=W_2\,\sigma(W_1x).$$

MoE 准备 $E$ 个 experts，router 计算 $p=\operatorname{softmax}(W_rx)$，只选择 top-k 集合 $S(x)$：

$$y=\sum_{i\in S(x)}\tilde p_iE_i(x).$$

若每个 expert 有 $P_e$ 个参数，则专家总参数约为 $EP_e$，单 token 激活约为 $kP_e$。但所有专家权重仍需存储；dispatch/combine、负载不均和跨卡通信也不会出现在这个简化参数公式里。

In [ ]:
import math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

np.set_printoptions(precision=3, suppress=True)  # 计算并保存当前步骤的中间状态。
rng = np.random.default_rng(7)  # 计算并保存当前步骤的中间状态。

def softmax(logits: np.ndarray, axis: int = -1) -> np.ndarray:  # 定义本节可复用的核心函数。
    shifted = logits - logits.max(axis=axis, keepdims=True)  # 计算并保存当前步骤的中间状态。
    exp = np.exp(shifted)  # 计算并保存当前步骤的中间状态。
    return exp / exp.sum(axis=axis, keepdims=True)  # 返回当前分支计算出的结果。

def relu(x: np.ndarray) -> np.ndarray:  # 定义本节可复用的核心函数。
    return np.maximum(x, 0.0)  # 返回当前分支计算出的结果。


## 2. 从零实现 top-k MoE 前向

下面每个 expert 都是独立的两层 MLP。输入形状为 `[tokens, d_model]`，router logits 为 `[tokens, n_experts]`。实现会记录每个 token 的专家 id、gate 权重和每个 expert 收到的 token 数。

教学代码用 Python 循环方便观察；生产实现会先按 expert 排序和分桶，形成 grouped GEMM，并在 expert parallel 场景中插入 all-to-all。

In [ ]:
class Expert:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model: int, d_hidden: int, rng: np.random.Generator):  # 定义本节可复用的核心函数。
        scale = 1 / math.sqrt(d_model)  # 计算并保存当前步骤的中间状态。
        self.w1 = rng.normal(0, scale, size=(d_model, d_hidden))  # 计算并保存当前步骤的中间状态。
        self.w2 = rng.normal(0, 1 / math.sqrt(d_hidden), size=(d_hidden, d_model))  # 计算并保存当前步骤的中间状态。

    def __call__(self, x: np.ndarray) -> np.ndarray:  # 定义本节可复用的核心函数。
        return relu(x @ self.w1) @ self.w2  # 返回当前分支计算出的结果。

class TopKMoE:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model: int, d_hidden: int, n_experts: int, top_k: int, rng):  # 定义本节可复用的核心函数。
        assert 1 <= top_k <= n_experts  # 用受控断言验证关键不变量。
        self.n_experts = n_experts  # 计算并保存当前步骤的中间状态。
        self.top_k = top_k  # 计算并保存当前步骤的中间状态。
        self.router = rng.normal(0, 1 / math.sqrt(d_model), size=(d_model, n_experts))  # 计算并保存当前步骤的中间状态。
        self.experts = [Expert(d_model, d_hidden, rng) for _ in range(n_experts)]  # 计算并保存当前步骤的中间状态。

    def __call__(self, x: np.ndarray):  # 定义本节可复用的核心函数。
        logits = x @ self.router  # 计算并保存当前步骤的中间状态。
        probabilities = softmax(logits)  # 计算并保存当前步骤的中间状态。
        # argpartition 取无序 top-k，再按概率从高到低排序，便于查看。
        top_ids = np.argpartition(probabilities, -self.top_k, axis=1)[:, -self.top_k:]  # 计算并保存当前步骤的中间状态。
        top_scores = np.take_along_axis(probabilities, top_ids, axis=1)  # 计算并保存当前步骤的中间状态。
        order = np.argsort(-top_scores, axis=1)  # 计算并保存当前步骤的中间状态。
        top_ids = np.take_along_axis(top_ids, order, axis=1)  # 计算并保存当前步骤的中间状态。
        top_scores = np.take_along_axis(top_scores, order, axis=1)  # 计算并保存当前步骤的中间状态。
        gates = top_scores / top_scores.sum(axis=1, keepdims=True)  # 计算并保存当前步骤的中间状态。

        output = np.zeros_like(x, dtype=float)  # 计算并保存当前步骤的中间状态。
        loads = np.zeros(self.n_experts, dtype=int)  # 计算并保存当前步骤的中间状态。
        for token_index in range(len(x)):  # 遍历输入元素以累积或检查结果。
            for route_index in range(self.top_k):  # 遍历输入元素以累积或检查结果。
                expert_id = int(top_ids[token_index, route_index])  # 计算并保存当前步骤的中间状态。
                output[token_index] += gates[token_index, route_index] * self.experts[expert_id](x[token_index])  # 计算并保存当前步骤的中间状态。
                loads[expert_id] += 1  # 计算并保存当前步骤的中间状态。
        trace = {"logits": logits, "probabilities": probabilities, "top_ids": top_ids, "gates": gates, "loads": loads}  # 计算并保存当前步骤的中间状态。
        return output, trace  # 返回当前分支计算出的结果。

# 6 个 token、4 维隐藏状态、4 个 experts、每 token 选 2 个。
x = rng.normal(size=(6, 4))  # 计算并保存当前步骤的中间状态。
moe = TopKMoE(d_model=4, d_hidden=8, n_experts=4, top_k=2, rng=rng)  # 计算并保存当前步骤的中间状态。
y, trace = moe(x)  # 计算并保存当前步骤的中间状态。
print("输入 shape:", x.shape, "输出 shape:", y.shape)  # 执行当前语句以推进本节示例。
print("top-k expert ids:\n", trace["top_ids"])  # 执行当前语句以推进本节示例。
print("归一化 gate:\n", trace["gates"])  # 执行当前语句以推进本节示例。
print("expert loads:", trace["loads"], "总路由数:", trace["loads"].sum())  # 执行当前语句以推进本节示例。


## 3. Capacity factor 与过载

有 $T$ 个 token、$E$ 个 experts、top-k 为 $k$ 时，若全部 top-k assignment 共用一个容量池，均匀负载期望是 $Tk/E$，固定容量可写为：

$$C=\left\lceil\text{capacity factor}\times\frac{Tk}{E}\right\rceil.$$

部分 top-2 实现会为第一、第二选择分别按 $T/E$ 分配容量，或把 $k$ 吸收到 factor 中；跨实现必须先统一口径。固定容量实现会把负载不足的 expert **padding 到 $C$ 个槽位**，以获得规则 shape；这会执行无效计算，但 padding 本身不能解决超载。某 expert 超过 $C$ 后，系统仍需选择：

- **drop / residual bypass**：省计算但可能损害质量；
- **reroute**：送往下一候选，改变路由规则并增加实现复杂度；
- **dropless / 动态容量**：全部处理，避免信息丢失但负载和尾延迟不规则。

下面模拟按 token 顺序接纳，超出容量的路由记为 dropped。

In [ ]:
def apply_capacity(top_ids: np.ndarray, n_experts: int, capacity_factor: float):  # 定义本节可复用的核心函数。
    tokens, top_k = top_ids.shape  # 计算并保存当前步骤的中间状态。
    capacity = math.ceil(capacity_factor * tokens * top_k / n_experts)  # 计算并保存当前步骤的中间状态。
    accepted = np.zeros_like(top_ids, dtype=bool)  # 计算并保存当前步骤的中间状态。
    loads = np.zeros(n_experts, dtype=int)  # 计算并保存当前步骤的中间状态。
    for token_index in range(tokens):  # 遍历输入元素以累积或检查结果。
        for route_index in range(top_k):  # 遍历输入元素以累积或检查结果。
            expert_id = int(top_ids[token_index, route_index])  # 计算并保存当前步骤的中间状态。
            if loads[expert_id] < capacity:  # 按当前条件选择后续控制路径。
                accepted[token_index, route_index] = True  # 计算并保存当前步骤的中间状态。
                loads[expert_id] += 1  # 计算并保存当前步骤的中间状态。
    return capacity, accepted, loads  # 返回当前分支计算出的结果。

for factor in [0.75, 1.0, 1.25, 2.0]:  # 遍历输入元素以累积或检查结果。
    capacity, accepted, loads = apply_capacity(trace["top_ids"], 4, factor)  # 计算并保存当前步骤的中间状态。
    dropped = accepted.size - int(accepted.sum())  # 计算并保存当前步骤的中间状态。
    print(f"factor={factor:>4} | capacity={capacity} | loads={loads} | dropped={dropped}")  # 计算并保存当前步骤的中间状态。


## 4. 为什么需要负载均衡损失

任务损失可能让少数 expert 早期获得更多 token 和梯度，形成“越常用越强、越强越常用”的正反馈，最终 expert collapse。经典辅助思路同时观察：

- $f_i$：实际派给 expert $i$ 的 token 比例；
- $P_i$：router 给 expert $i$ 的平均概率。

一种常见形式与 $E\sum_i f_iP_i$ 同阶；分配和概率都集中时惩罚变大。具体论文和实现的归一化、top-k 统计与系数不同，不能把示意公式当作所有框架的唯一标准。

辅助系数太小压不住热点，太大又可能为了均匀而干扰语义路由。新方法也会用动态 bias 等方式实现不直接加入训练 loss 的均衡。

In [ ]:
def load_balance_measure(probabilities: np.ndarray, top_ids: np.ndarray, n_experts: int) -> float:  # 定义本节可复用的核心函数。
    # 这里以 top-1 分配演示 f_i；top-2 或分组路由需按实现重新定义。
    top1 = top_ids[:, 0]  # 计算并保存当前步骤的中间状态。
    fractions = np.bincount(top1, minlength=n_experts) / len(top1)  # 计算并保存当前步骤的中间状态。
    mean_probability = probabilities.mean(axis=0)  # 计算并保存当前步骤的中间状态。
    return float(n_experts * np.sum(fractions * mean_probability))  # 返回当前分支计算出的结果。

def routing_metrics(loads: np.ndarray) -> dict[str, float]:  # 定义本节可复用的核心函数。
    loads = loads.astype(float)  # 计算并保存当前步骤的中间状态。
    total = loads.sum()  # 计算并保存当前步骤的中间状态。
    probs = loads / total if total else np.zeros_like(loads)  # 计算并保存当前步骤的中间状态。
    positive = probs[probs > 0]  # 计算并保存当前步骤的中间状态。
    entropy = float(-np.sum(positive * np.log(positive)))  # 计算并保存当前步骤的中间状态。
    normalized_entropy = entropy / math.log(len(loads)) if len(loads) > 1 else 1.0  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "max_over_mean": float(loads.max() / loads.mean()) if loads.mean() else 0.0,  # 执行当前语句以推进本节示例。
        "normalized_entropy": normalized_entropy,  # 执行当前语句以推进本节示例。
        "unused_experts": int(np.sum(loads == 0)),  # 计算并保存当前步骤的中间状态。
    }  # 执行当前语句以推进本节示例。

print("均衡度量（越接近均匀基线越好，非通用 loss）:", load_balance_measure(trace["probabilities"], trace["top_ids"], 4))  # 执行当前语句以推进本节示例。
print("路由指标:", routing_metrics(trace["loads"]))  # 执行当前语句以推进本节示例。


## 5. top-1、top-2 与梯度

`TopK` 的专家集合选择是离散的；标准实现中，任务梯度沿已选 expert 和已选 gate 权重传播，未选 expert 对该 token 通常没有任务梯度。router noise/jitter 可在训练早期增加探索；z-loss 约束 router 的 `logsumexp` 数值尺度，但它不是负载均衡或熵正则，不能保证 top-k 分布均匀。

- top-1：专家计算和通信更低，但单一路由错误影响更直接。
- top-2：两个 expert 加权组合，通常更平滑、有冗余，但路由槽位、专家 FLOPs 和 dispatch 流量明显增加。

训练 top-2 后推理时直接改 top-1 会改变模型函数；除非有相应训练或校准，不能把它当作无损开关。

## 6. Expert Parallel 与 all-to-all

若每张 GPU 只保存部分 experts，一个 token 的目标 expert 很可能在别的设备。一次 MoE 层通常需要：

```text
本地 token -> 按 expert 分桶 -> all-to-all dispatch
          -> 各卡执行本地 experts -> all-to-all combine
          -> 恢复原 batch/sequence 顺序
```

这与其他并行方式的关注点不同：

- data parallel 复制模型、切 batch；
- tensor parallel 切单个矩阵；
- pipeline parallel 切层；
- expert parallel 切 experts，并随 token 路由动态通信。

理论 FLOPs 相同也可能因跨节点链路、热点 expert、小而不规则的 GEMM 和同步等待产生很不同的 P99 延迟。专家放置应结合实际路由共现与网络拓扑，而不是只按数量平均分配。

In [ ]:
def moe_parameter_estimate(  # 定义本节可复用的核心函数。
    layers: int, d_model: int, d_ff: int, experts_per_layer: int, top_k: int  # 执行当前语句以推进本节示例。
) -> dict[str, int]:  # 执行当前语句以推进本节示例。
    # 简化为普通两层 FFN，SwiGLU 会有不同常数；不含 attention/embedding/router。
    per_expert = 2 * d_model * d_ff  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "每个expert参数": per_expert,  # 执行当前语句以推进本节示例。
        "全部MoE层专家总参数": layers * experts_per_layer * per_expert,  # 执行当前语句以推进本节示例。
        "每token激活专家参数_跨层求和": layers * top_k * per_expert,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

estimate = moe_parameter_estimate(layers=24, d_model=2048, d_ff=5632, experts_per_layer=64, top_k=2)  # 计算并保存当前步骤的中间状态。
for key, value in estimate.items():  # 遍历输入元素以累积或检查结果。
    print(f"{key}: {value / 1e9:.3f} B")  # 执行当前语句以推进本节示例。
print("注意：真实模型还必须加 dense attention、embedding、router、norm 和 shared experts。")  # 执行当前语句以推进本节示例。


## 7. 生产监控与面试总结

每层至少监控 expert token 数、gate 概率、最大/平均负载、路由熵、未使用专家、drop/reroute/padding 比例、router logits 与梯度、all-to-all 时间和最慢 expert 时间；并按语言、任务与请求类型分桶。负载均匀不等于专业化合理，专业化也不等于线上高效，必须同时看 loss、下游质量、吞吐和 P99。

### 面试速答

`MoE 用 router 为每个 token 只激活 top-k experts，因此总参数随专家数增长，单 token 专家计算主要随 k 增长。代价是全部专家仍需存储，还要解决负载均衡、容量、离散路由和 expert-parallel all-to-all；所以理论 FLOPs 不能直接等同真实延迟。`

### 练习

1. 给 `TopKMoE` 增加 shared expert，并比较总参数与激活参数。
2. 实现 capacity 满后改投下一候选 expert，而不是直接 drop。
3. 构造所有 token 都偏向 expert 0 的 router，观察熵和最大/平均负载。
4. 将 top-2 改为 top-1，比较输出、路由数和容量。
5. 设计一个包含 P50/P99 与 all-to-all 字节数的线上 MoE 压测表。